# Hafta 2 — Mekânsal Eşleştirme + Tahliye Skoru + Yayılma Hızı
**Sultan** — AteşKes (EERİS+)

Girdi olarak yangın noktası + kritik alan listesi (Hafta 1 çıktısı) alan, çıktı olarak `tahliye_skoru` ve yayılma hızı üreten fonksiyon seti.

In [1]:
import pandas as pd
from hesaplamalar import (
    en_yakin_kritik_alan, tahliye_skoru, egim_yonu_belirle,
    ruzgar_egime_uyumlu_mu, yayilma_hizi_belirle,
)

kritik_alanlar_df = pd.read_csv("hafta1_ornek_veri.csv")
kritik_alanlar = kritik_alanlar_df.to_dict("records")
kritik_alanlar_df

,bolge_id,isim,tip,lat,lon,yukseklik_metre,egim_derece,risk_skoru,oncelik_skoru
0,mugla_01,İslamhaneleri,koy,37.029895,27.294703,40.0,11.9,0.141,0.7
1,mugla_02,Bodrum Amerikan Hastanesi,hastane,37.039939,27.428962,13.0,1.7,0.105,1.0
2,mugla_03,Kemer İlköğretim Okulu,okul,36.646920,29.362071,124.0,1.1,0.103,0.9
3,mugla_04,İkinci Bahar Huzur Evi,huzurevi,36.855276,28.279101,5.0,9.1,0.131,0.9
4,mugla_05,Gürece,koy,37.039538,27.322432,160.0,10.8,0.137,0.7
5,mugla_06,isimsiz,hastane,36.622309,29.115010,5.0,2.3,0.108,1.0


## Mock yangın noktaları
**NOT:** FIRMS entegrasyonu (`/yangin-noktalari`) Esma'nın Hafta 1 işi. O hazır olana kadar bu fonksiyon setini test etmek için mock yangın noktaları kullanıyoruz. Fonksiyonun girdi imzası (`lat`, `lon`, `ruzgar_hizi`, `ruzgar_yonu_derece` içeren kayıt listesi) FIRMS+Open-Meteo birleşiminden gelecek gerçek veriyle aynı olacağı için entegrasyonda sadece veri kaynağı değişecek, fonksiyon değişmeyecek.

In [2]:
mock_yangin_noktalari = [
    {"yangin_id": "yangin_01", "lat": 37.03, "lon": 27.30, "ruzgar_hizi": 25, "ruzgar_yonu_derece": 0},
    {"yangin_id": "yangin_02", "lat": 36.85, "lon": 28.28, "ruzgar_hizi": 15, "ruzgar_yonu_derece": 180},
]

## Hafta 2 teslimi: yangin_kritik_alan_eslestir()
Haversine ile en yakın kritik alanı bulur, `tahliye_skoru = risk × oncelik` hesaplar ve rüzgar/eğim ilişkisine göre yayılma hızını belirler.

In [3]:
def yangin_kritik_alan_eslestir(yangin_noktalari, kritik_alanlar):
    sonuclar = []
    for yangin in yangin_noktalari:
        alan, mesafe = en_yakin_kritik_alan(yangin["lat"], yangin["lon"], kritik_alanlar)
        if alan is None:
            continue
        tahliye = tahliye_skoru(alan["risk_skoru"], alan["oncelik_skoru"])

        # Basitleştirme: gerçek 5-noktalı yükseklik verisi bu aşamada elde
        # tutulmadığından (sadece egim_derece saklandı), yön uyumu kaba bir
        # kural ile kestiriliyor. Rota/yol modülüne bağlanırken gerçek
        # egim_yonu_belirle() + ruzgar_egime_uyumlu_mu() zincirine geçilecek.
        uyumlu = yangin["ruzgar_yonu_derece"] < 90
        hiz = yayilma_hizi_belirle(yangin["ruzgar_hizi"], alan["egim_derece"], uyumlu)

        sonuclar.append({
            "yangin_id": yangin["yangin_id"],
            "en_yakin_kritik_alan": alan["isim"],
            "kritik_alan_tipi": alan["tip"],
            "mesafe_metre": round(mesafe, 1),
            "tahliye_skoru": tahliye,
            "yayilma_hizi": hiz,
        })
    return sonuclar


sonuc = yangin_kritik_alan_eslestir(mock_yangin_noktalari, kritik_alanlar)
sonuc_df = pd.DataFrame(sonuc)
sonuc_df

,yangin_id,en_yakin_kritik_alan,kritik_alan_tipi,mesafe_metre,tahliye_skoru,yayilma_hizi
0,yangin_01,İslamhaneleri,koy,470.4,0.099,yavas
1,yangin_02,İkinci Bahar Huzur Evi,huzurevi,592.1,0.118,yavas


In [4]:
sonuc_df.to_csv("hafta2_eslesme_sonuc.csv", index=False, encoding="utf-8")
print("Kaydedildi: hafta2_eslesme_sonuc.csv")

Kaydedildi: hafta2_eslesme_sonuc.csv


## Notlar / sonraki adım
- `hesaplamalar.py` artık hem Hafta 1 hem Hafta 2 fonksiyonlarını içeriyor — Esma bu modülü olduğu gibi backend'e import edebilir.
- Mock yangın noktaları, Esma'nın `/yangin-noktalari` endpoint'i hazır olunca gerçek FIRMS verisiyle değiştirilecek (fonksiyon imzası aynı kalıyor).
- Hafta 3'te A* rota maliyetine `egim_cezasi()` eklenecek; bu modüldeki `egim_derece` alanı doğrudan orada kullanılabilir.